# preprocessing

In [ ]:
import re

# 1. TỪ ĐIỂN MAPPING (SLANG & TEENCODE)
SLANG_MAP = {
    # --- 1. NHÓM TĂNG CƯỜNG ĐỘ (INTENSITY) ---
    # Dùng để nhấn mạnh cảm xúc (thường đứng sau tính từ)
    "vcl": "<INTENSITY>", "vl": "<INTENSITY>", "vlon": "<INTENSITY>",
    "vcđ": "<INTENSITY>", "vcd": "<INTENSITY>", "vch": "<INTENSITY>",
    "vãi": "<INTENSITY>", "vai": "<INTENSITY>", "vai~": "<INTENSITY>",
    "vail": "<INTENSITY>", "vailon": "<INTENSITY>", "vailoz": "<INTENSITY>",
    "vlin": "<INTENSITY>", "vlinz": "<INTENSITY>", "vler": "<INTENSITY>",
    "vcb": "<INTENSITY>", "vcc": "<INTENSITY>", "vc": "<INTENSITY>",
    "kinh": "<INTENSITY>", "khung": "<INTENSITY>", "phe": "<INTENSITY>",
    "ba dao": "<INTENSITY>", "vip": "<INTENSITY>", "pro": "<INTENSITY>",
    "ỉnh": "<INTENSITY>", # đỉnh
    "ao nhay": "<INTENSITY>", # ảo nhỉ
    "ao ma": "<INTENSITY>", # ảo ma

    # --- 2. NHÓM THÔ TỤC (PROFANITY) ---
    # Các từ chửi thề mang tính cảm thán, đệm câu
    "dm": "<PROFANITY>", "dmm": "<PROFANITY>", "duma": "<PROFANITY>", "dume": "<PROFANITY>",
    "đm": "<PROFANITY>", "đmm": "<PROFANITY>", "đcm": "<PROFANITY>", "dcm": "<PROFANITY>",
    "dkm": "<PROFANITY>", "dkmm": "<PROFANITY>", "dik": "<PROFANITY>",
    "clm": "<PROFANITY>", "cl": "<PROFANITY>", "clgt": "<PROFANITY>", "clgv": "<PROFANITY>",
    "cc": "<PROFANITY>", "ccc": "<PROFANITY>", "ccmm": "<PROFANITY>",
    "cak": "<PROFANITY>", "kak": "<PROFANITY>", "kăk": "<PROFANITY>",
    "loz": "<PROFANITY>", "lon": "<PROFANITY>", "lìn": "<PROFANITY>",
    "ml": "<PROFANITY>", "mml": "<PROFANITY>", "clmm": "<PROFANITY>",
    "cdm": "<PROFANITY>",
    "đéo": "<PROFANITY>", "deo": "<PROFANITY>", "dell": "<PROFANITY>", "dek": "<PROFANITY>",
    "éo": "<PROFANITY>", "kẹc": "<PROFANITY>",
    "fuck": "<PROFANITY>", "fuk": "<PROFANITY>", "shit": "<PROFANITY>",
    "bitch": "<PROFANITY>", "dit": "<PROFANITY>", "djt": "<PROFANITY>",
    "ditme": "<PROFANITY>", "ditmom": "<PROFANITY>",
}

TEENCODE_MAP = {
    # --- 1. NHÓM PHỦ ĐỊNH & KHẲNG ĐỊNH ---
    "k": "không", "ko": "không", "kh": "không", "hok": "không",
    "hong": "không", "hông": "không", "hem": "không", "nô": "không",
    "khog": "không", "khong": "không", "kg": "không", "khg": "không",
    "k0": "không", "kô": "không", "no": "không", "nop": "không",
    "chx": "chưa", "chưa": "chưa", "chư": "chưa",
    "oke": "được", "ok": "được", "oki": "được", "okie": "được",
    "uk": "ừ", "ukm": "ừ", "uhm": "ừ", "uh": "ừ", "ye": "vâng", "yep": "vâng",
    "r": "rồi", "roi": "rồi", "rùi": "rồi", "ròi": "rồi", "oy": "rồi",

    # --- 2. ĐẠI TỪ / XƯNG HÔ (Stream Context) ---
    "mik": "mình", "mk": "mình", "mjk": "mình", "t": "tôi", "tao": "tôi",
    "tui": "tôi", "toi": "tôi", "tớ": "tôi",
    "b": "bạn", "u": "bạn", "m": "mày", "may": "mày", "mem": "thành viên",
    "ck": "chồng", "vk": "vợ", "bx": "bà xã", "ox": "ông xã",
    "ad": "quản trị viên", "add": "quản trị viên", "admin": "quản trị viên",
    "ae": "anh em", "ace": "anh chị em", "pr": "bạn", "pro": "người giỏi",
    "bro": "anh bạn", "sir": "ông", "idol": "thần tượng",
    "fan": "người hâm mộ", "fanti": "người hâm mộ trêu chọc",
    "sp": "sư phụ", "su phu": "sư phụ",

    # --- 3. TỪ NGỮ GAME & LIVESTREAM (Quan trọng cho Sentiment) ---
    # Chuyển đổi về từ mang sắc thái rõ ràng để Bert hiểu
    "game": "trò chơi", "gem": "trò chơi", "gêm": "trò chơi",
    "lag": "giật", "lac": "giật", "laggg": "giật", "lắc": "giật",
    "win": "thắng", "w": "thắng", "ez": "dễ dàng", "easy": "dễ dàng",
    "lose": "thua", "luz": "thua",
    "gg": "đầu hàng",
    "rank": "xếp hạng", "ranh": "xếp hạng",
    "top": "đường trên", "mid": "đường giữa", "bot": "đường dưới", "sp": "hỗ trợ",
    "gank": "tấn công bất ngờ", "ks": "cướp công", "camp": "cắm trại",
    "feed": "chơi tệ", "feeeed": "chơi quá tệ", "gà": "kém", "ga": "kém", "noob": "kém",
    "pro": "giỏi", "vip": "xuất sắc", "đỉnh": "xuất sắc",
    "mvp": "người chơi hay nhất",
    "afk": "treo máy", "dis": "mất kết nối", "out": "thoát",
    "hack": "gian lận", "cheat": "gian lận", "buff": "tăng sức mạnh", "nerf": "giảm sức mạnh",
    "ban": "cấm", "pick": "chọn",
    "mic": "nghe", "cam": "camera",
    "live": "trực tiếp", "stream": "trực tiếp", "lai": "trực tiếp",
    "sub": "đăng ký", "sup": "đăng ký", "subcribe": "đăng ký", "đk": "đăng ký",
    "like": "thích", "share": "chia sẻ", "view": "lượt xem",
    "donate": "ủng hộ tiền", "donết": "ủng hộ tiền", "hiến máu": "ủng hộ tiền",
    "mod": "người kiểm duyệt", 'goat' : 'hay nhất' , 'peak' : 'đỉnh' , 'as' : 'ánh sáng' , 'kc' : 'kim cương' , 'heal' : 'hồi máu' , 'wtf' : 'cái gì thế' , 'turn' : 'lượt',
    'dnt' : 'ủng hộ tiền' , 'fact' : 'sự thật' , 'true' : 'sự thật',

    # --- 4. HÀNH ĐỘNG / TRẠNG THÁI / CẢM XÚC ---
    "dc": "được", "đc": "được", "dk": "được", "đk": "được", "ddc": "được", # ddc là lỗi unikey
    "bjk": "biết", "bit": "biết", "biêt": "biết", "bik": "biết", "bít": "biết",
    "nt": "nhắn tin", "ib": "nhắn tin", "inb": "nhắn tin", "pm": "nhắn tin",
    "rep": "trả lời", "tl": "trả lời",
    "thik": "thích", "thix": "thích", "tk": "thích",
    "iu": "yêu", "ju": "yêu", "yeu": "yêu",
    "h": "giờ", "bh": "bây giờ", "hjo": "bây giờ", "bjo": "bây giờ",
    "hnay": "hôm nay", "nay": "hôm nay",
    "trc": "trước", "truoc": "trước",
    "sau": "sau",
    "nx": "nữa", "nua": "nữa",
    "xem": "xem", "xm": "xem",
    "lm": "làm", "lam": "làm",
    "chuc": "chúc",
    "ngon": "ngon",
     "g9": "ngủ ngon",
    "cuoi": "cười", "cừi": "cười",
    "khoc": "khóc",
    "buon": "buồn",
    "so": "sợ",
    "met": "mệt",


    # --- 5. TỪ ĐỆM / CÂU HỎI / KHÁC ---
    "j": "gì", "gi": "gì", "ji": "gì", "zi": "gì",
    "z": "vậy", "v": "vậy", "zậy": "vậy", "dậy": "vậy", "vay": "vậy",
    "s": "sao", "shao": "sao",
    "wa": "quá", "qa": "quá", "qua": "quá", "woa": "quá",
    "thui": "thôi", "thoi": "thôi",
    "lun": "luôn", "luon": "luôn",
    "ah": "à", "uh": "ừ",
    "tks": "cảm ơn", "thank": "cảm ơn", "thanks": "cảm ơn", "tk": "cảm ơn", "cam on": "cảm ơn",
    "vs": "với", "voi": "với",
    "choy": "chơi", "choi": "chơi",
    "nhieu": "nhiều", "nhiu": "nhiều",
    "it": "ít",
    "nhanh": "nhanh",
    "cham": "chậm",

    # --- 6. VIẾT TẮT PHỔ BIẾN KHÁC (Common Abbreviations) ---
    "klq": "không liên quan",
    "gt": "giới thiệu",
    "kb": "kết bạn",
    "nc": "nói chuyện",
    "bl": "bình luận", "cmt": "bình luận", "comment": "bình luận",
    "stt": "trạng thái",
    "sn": "sinh nhật",
    "hp": "hạnh phúc",
    "cj": "chị",
    "tr": "trời", "troi": "trời", "trùi": "trời",
    "đt": "điện thoại",
    "mt": "máy tính",
    "pc": "máy tính",
    "fb": "facebook",
    "ytb": "youtube",
    "ms": "mới", "moi": "mới",
    'trôn' : 'giỡn', 'tron' : 'giỡn',

    # --- 7. LỖI GÕ UNIKEY / TELEX PHỔ BIẾN (Rất quan trọng) ---
    "ddung": "đúng", "dungs": "đúng",
    "saai": "sai",
    "ddau": "đâu", "ddaau": "đâu",
    "khiu": "khó chịu",
    "tnao": "thế nào", "ntn": "như thế nào",
    "gê": "ghê", "ghe": "ghê",
}

# HÀM BỔ TRỢ ĐỂ XỬ LÝ SỐ "2" (Vừa là số, vừa là "hi", vừa là "thứ 2")
# Lưu ý: Dictionary không xử lý ngữ cảnh, nên chỉ map những từ chắc chắn.
ENGLISH_GAMER_MAP = {
    # =================================================================
    # 1. GENERAL GAMING TERMS (Dùng chung cho cả 3 game)
    # =================================================================
    # Tích cực / Khen ngợi
    "gg": "trận đấu hay", "ggwp": "chơi hay lắm", "good job": "làm tốt lắm",
    "nice": "tốt", "nai": "tốt", "nais": "tốt", # Biến thể âm đọc
    "pro": "người chơi giỏi", "vip": "xuất sắc",
    "carry": "gánh team", "gánh": "gánh team", "car": "gánh team",
    "ez": "dễ dàng", "izi": "dễ dàng", "easy": "dễ dàng",
    "win": "thắng", "victory": "chiến thắng",
    "cb": "lật kèo", "comeback": "lật kèo", # Rất quan trọng trong sentiment

    # Tiêu cực / Chỉ trích (Ngoài bộ SLANG_MAP cũ)
    "noob": "người chơi kém", "nub": "người chơi kém", "newbie": "người mới",
    "trash": "rác", "tras": "rác", # Chỉ kỹ năng tồi
    "bad": "tệ",
    "throw": "quăng game", "quang game": "cố tình thua", # Hành động phá hoại
    "feed": "hiến mạng", "feeder": "kẻ hiến mạng", "fid": "hiến mạng",
    "afk": "treo máy", "out": "thoát trận", "quit": "bỏ cuộc",
    "report": "tố cáo", "rp": "tố cáo", "ripot": "tố cáo",
    "sv": "người chơi tồi", 'troll' : 'giỡn' , 'cook' : 'bị loại', 'cut' : 'bị loại',
    'var' : 'đụng chạm', 'job' : 'quảng cáo',

    # Kỹ thuật / Mạng
    "lag": "giật", "lac": "giật", "delay": "trễ", "ping": "độ trễ",
    "bug": "lỗi game", "glitch": "lỗi hiển thị",
    "fix": "sửa lỗi", "buff": "tăng sức mạnh", "nerf": "giảm sức mạnh",
    "meta": "chiến thuật hiệu quả nhất", 'damage' : 'sát thương' , 'no' : 'không',

    # =================================================================
    # 2. TFT - ĐẤU TRƯỜNG CHÂN LÝ (AUTO CHESS)
    # =================================================================
    # Hành động
    "roll": "xoay tướng", "reroll": "xoay tướng liên tục",
    "slowroll": "xoay tướng chậm", "hyperroll": "xoay tướng nhanh",
    "form": "đội hình", "comp": "đội hình", "bai": "bài",
    "pivot": "xoay bài", "xoay": "đổi đội hình",
    "check": "kiểm tra", "scout": "soi bài",
    "flex": "chơi linh hoạt",
    "slam": "ghép đồ sớm", "xep": "xếp đội hình", 'spam' : 'lặp lui lặp tới',

    # Kinh tế & Cấp độ
    "eco": "tích tiền", "economy": "kinh tế",
    "rush": "lên cấp nhanh", "rush 8": "lên cấp 8 nhanh", "level": "cấp độ",
    "all in": "xả hết tiền", "xoay het": "xả hết tiền", 'lsd' :'lịch sử đấu',

    # May rủi (Sentiment cực mạnh trong TFT)
    "luck": "may mắn", "lucky": "may mắn", "hên": "may mắn",
    "unluck": "xui xẻo", "den": "xui xẻo",
    "nhan pham": "may mắn", "dog": "chó đỏ", # Ám chỉ may mắn tột độ
    "high roll": "may mắn cực độ", "low roll": "xui xẻo cực độ",
    "top 1": "hạng nhất", "top 8": "hạng bét", "cút": "bị loại",

    # =================================================================
    # 3. LIÊN QUÂN MOBILE / MOBA
    # =================================================================
    # Vị trí (Role)
    "ad": "xạ thủ", "adc": "xạ thủ",
    "ap": "pháp sư", "mid": "đường giữa",
    "sp": "hỗ trợ", "sup": "hỗ trợ", "support": "hỗ trợ",
    "top": "đường trên", "caesar": "đường tà thần",
    "jung": "đi rừng", "jg": "đi rừng", "rung": "đi rừng",

    # Hành động chiến đấu
    "gank": "đi bắt lẻ", "ganh": "đi bắt lẻ",
    "farm": "kiếm vàng", "fam": "kiếm vàng",
    "push": "đẩy trụ", "pus": "đẩy trụ",
    "def": "phòng thủ", "thu": "phòng thủ", "backdoor": "đẩy trộm",
    "combat": "giao tranh", "cb": "giao tranh", "figh": "giao tranh",
    "skill": "kỹ năng", "skil": "kỹ năng", "chieu": "chiêu",
    "ulti": "chiêu cuối", "untl": "chiêu cuối", "until": "chiêu cuối",
    "combo": "chuỗi chiêu",

    # Trạng thái
    "stun": "choáng", "stune": "choáng",
    "slow": "làm chậm",
    "silence": "câm lặng",
    "one shot": "hạ gục nhanh", "soc dame": "sốc sát thương",
    "ks": "cướp mạng", "steal": "cướp",
    "snowball": "lăn cầu tuyết",

    # =================================================================
    # 4. PUBG / FPS / SURVIVAL
    # =================================================================
    # Súng & Đạn
    "awm": "súng ngắm mạnh", "kar": "súng ngắm", "k98": "súng ngắm",
    "scope": "ống ngắm", "cop": "ống ngắm", "x4": "ống ngắm 4x",
    "ammo": "đạn", "dan": "đạn",
    "recoil": "độ giật", "ghim tam": "ghìm tâm",
    "tap": "bắn từng viên", "say": "sấy", "spray": "sấy",

    # Vật phẩm
    "loot": "nhặt đồ", "lut": "nhặt đồ",
    "map": "bản đồ",
    "bo": "vòng bo", "zone": "vòng bo", "blue": "vòng xanh",
    "med": "thuốc", "mau": "máu", "first aid": "bộ cứu thương",
    "mu": "mũ", "giap": "giáp", "vest": "giáp",
    "thinh": "hòm tiếp tế", "drop": "hòm tiếp tế", "airdrop": "hòm tiếp tế",

    # Hành động
    "camp": "núp lùm", "camper": "kẻ núp lùm",
    "rush": "tấn công nhanh", "cong": "công nhà",
    "cover": "bảo kê", "ke": "kê súng",
    "revive": "cứu", "cuu": "cứu", "hoi sinh": "hồi sinh",
    "knock": "bị gục", "guc": "bị gục", "noc": "bị gục",
    "clear": "dọn sạch",
    "check": "kiểm tra",
    'firm' : "hạ gục", 'phơm' : 'hạ gục' , 'phom' : 'hạ gục',
    "tpp": "góc nhìn thứ 3", "fpp": "góc nhìn thứ 1" ,
    'line up' : 'đội hình' , 'lineup' : 'đội hình', 'backup' : 'hổ trợ' , 'back up' : 'hổ trợ'
}

In [ ]:
# 2. CÁC HÀM XỬ LÝ PHỤ TRỢ
def normalize_slang(text):
    """Chuyển đổi các từ lóng/chửi thề thành token đặc biệt"""
    for slang, tag in SLANG_MAP.items():
      last_char = slang[-1]
      pattern = rf'\b({re.escape(slang)}){last_char}*\b'
      text = re.sub(pattern, tag, text, flags=re.IGNORECASE)
    return text


def normalize_teencode(text):
    """Chuyển đổi teencode thành tiếng Việt chuẩn"""
    words = text.split()
    return " ".join(TEENCODE_MAP.get(w, w) for w in words)

def normalize_emoji(text):
    """Chuyển đổi các ký tự emoji thành token cảm xúc"""
    patterns = {
        # LAUGH :)) =)) :))) xD
        r"([:=-]-?\)+)": " <LAUGH> ",
        r"([:=-]-?\]+)": " <LAUGH> ",
        r"(xD+|XD+|XDD+)": " <LAUGH> ",
        r"(\^\^|\^_\^)": " <LAUGH> ",
        r"(:>+)": " <LAUGH> ",
        r"(haha+)": " <LAUGH> ",
        r"(hihi+)": " <LAUGH> ",
        r"(kk+)": " <LAUGH> ",

        # SAD :( =( :(( =((
        r"([:=]-?\(+)+": " <SAD> ",
        r"(:<+)": " <SAD> ",
        r"(T_T|TT_TT|;_;|ToT)": " <SAD> ",
        r":'\(": " <SAD> ",
        r"(huhu+)": " <SAD> ",
        r"(hix+)": " <SAD> ",
    }
    for p, rpl in patterns.items():
        text = re.sub(p, rpl, text, flags=re.IGNORECASE)
    return text

def normalize_english_gamers(text):
    """
    Mapping các thuật ngữ tiếng Anh trong game về tiếng Việt
    để Model ViSoBERT hiểu ngữ cảnh tốt hơn.
    """
    words = text.split()
    # Sử dụng .lower() để đảm bảo khớp key trong dict
    return " ".join([ENGLISH_GAMER_MAP.get(word.lower(), word) for word in words])

# 3. PIPELINE CHÍNH (Đã sắp xếp thứ tự)
def cleaning_pipeline(text):
    if not text or not isinstance(text, str):
        return ""

    # BƯỚC 1: Xử lý cơ bản & URL
    text = text.lower() # Chuyển về chữ thường trước tiên
    text = re.sub(r'http\S+|www\S+|https\S+', '', text) # Xóa URL
    text = re.sub(r'@\w+', '', text) # Xóa mention (tag)

    # BƯỚC 2: Xử lý Emoji (Trước khi xóa ký tự đặc biệt)
    text = normalize_emoji(text)

    # BƯỚC 3: Xử lý kéo dài ký tự (Character Flooding)
    # Ví dụ: "đẹpppppp" -> "đẹp" (Lưu ý: nên đưa về 1 ký tự để map teencode chính xác hơn)
    # Nếu muốn giữ sắc thái cảm xúc, có thể dùng \1\1 nhưng sẽ khó map dictionary.
    text = re.sub(r'([a-zđăâêôơưáàảãạắằẳẵặấầẩẫậéèẻẽẹếềểễệíìỉĩịóòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵ])\1+', r'\1\1', text)
    # 3. Bảo vệ các emoji kéo dài (ví dụ: 😍😍😍 -> 😍)
    # Chúng ta chỉ muốn 1 emoji để giảm nhiễu cho mô hình
    text = re.sub(r'(\U00010000-\U0010ffff)\1+', r'\1', text)

    # 4. LỌC NHIỄU NHƯNG GIỮ EMOJI
    # Giữ lại: Chữ (\w), Khoảng trắng (\s), Token (<>), Dấu câu cảm xúc (!?), và Emoji
    text = re.sub(r'[^\w\s<>!\?\-+\()\[]\U00010000-\U0010ffff]', ' ', text)

    # BƯỚC 5: Xử lý khoảng trắng thừa lần 1 (để tách từ cho chính xác)
    text = re.sub(r'\s+', ' ', text).strip()

    # BƯỚC 6: Mapping Từ điển (Teencode & Slang)
    # Lưu ý: Teencode nên chạy trước hoặc cùng lúc.
    # Ở đây ta chạy tuần tự.
    text = normalize_english_gamers(text)

    text = normalize_teencode(text)

    text = normalize_slang(text)

    # BƯỚC 7: Xử lý lặp từ (Word Flooding)
    # Ví dụ: "alo alo alo" -> "alo"
    text = re.sub(r'\b(\w+)( \1\b)+', r'\1', text)

    # BƯỚC 8: Làm sạch token tag
    # Đảm bảo các token như <LAUGH> không bị dính dấu cách thừa bên trong (vd: < LAUGH >)
    text = re.sub(r'<\s*(\w+)\s*>', r'<\1>', text)

    # BƯỚC 9: Final Trim
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# --- TEST ---
if __name__ == "__main__":
    examples = [
        "Hàng về đẹp vcl lun ae ơi!!! :))) http://shopee.vn",
        "Ko thể tin dc, sp như loz dmm",
        "Buồn quá huhuu T_T",
        "Hayyyyyyy quáaaaaaa bạn uiiiii",
        "ib mình nha b, tks b nhìu",
        " ko phải con mình , mà xem còn thấy đau như vậy huống gì người trong cuộc . thật là phẫn nộ mà . cơ quan chức năng làm việc quá chậm trễ , đến giờ mà vẫn chưa tìm ra người chịu trách nhiệm . 😠😠😠😠😠😠"
    ]

    print("--- KẾT QUẢ ---")
    for ex in examples:
        print(f"Original: {ex}")
        print(f"Cleaned : {cleaning_pipeline(ex)}\n")

--- KẾT QUẢ ---
Original: Hàng về đẹp vcl lun ae ơi!!! :))) http://shopee.vn
Cleaned : hàng về đẹp <INTENSITY> luôn anh em ơi!!! <LAUGH>

Original: Ko thể tin dc, sp như loz dmm
Cleaned : không thể tin dc, hỗ trợ như <PROFANITY> <PROFANITY>

Original: Buồn quá huhuu T_T
Cleaned : buồn quá <SAD> <SAD>

Original: Hayyyyyyy quáaaaaaa bạn uiiiii
Cleaned : hayy quáaa bạn uii

Original: ib mình nha b, tks b nhìu
Cleaned : nhắn tin mình nha b, cảm ơn bạn nhìu

Original:  ko phải con mình , mà xem còn thấy đau như vậy huống gì người trong cuộc . thật là phẫn nộ mà . cơ quan chức năng làm việc quá chậm trễ , đến giờ mà vẫn chưa tìm ra người chịu trách nhiệm . 😠😠😠😠😠😠
Cleaned : không phải con mình , mà xem còn thấy đau như vậy huống gì người trong cuộc . thật là phẫn nộ mà . cơ quan chức năng làm việc quá chậm trễ , đến giờ mà vẫn chưa tìm ra người chịu trách nhiệm . 😠😠😠😠😠😠



In [ ]:
import pandas as pd


df = pd.read_csv('/content/final_data_chat.csv')
df.head()

,message,emotion,toxicity,interaction_type
0,Hello e chè,Neutral,Clean,reaction
1,ddaauf,Neutral,Clean,other
2,zoooo,Excitement,Clean,reaction
3,ô,Neutral,Clean,reaction
4,lo,Neutral,Clean,reaction


In [ ]:
df['emotion'].unique()

array(['Neutral', 'Excitement', 'Frustration', 'Disappointment',
       'viewer_request'], dtype=object)

In [ ]:
df = df.drop_duplicates()
df = df[df['emotion'] != 'viewer_request']
df.head()

,message,emotion,toxicity,interaction_type
0,Hello e chè,Neutral,Clean,reaction
1,ddaauf,Neutral,Clean,other
2,zoooo,Excitement,Clean,reaction
3,ô,Neutral,Clean,reaction
4,lo,Neutral,Clean,reaction


In [ ]:
for s in df['message'].values[-200:]:

    print(f"Original: {s}")

    print(f"Cleaned : {cleaning_pipeline(s)}\n")

Original: ai vắng mặt thì bị nói xấu thế thou
Cleaned : ai vắng mặt thì bị nói xấu thế thou

Original: bú đúng mà tại anh gà
Cleaned : bú đúng mà tại anh kém

Original: nam sau tq len ngoi
Cleaned : nam sau tq len ngoi

Original: Đừng chơi vs ảnh nữa mass ơi
Cleaned : đừng chơi với ảnh nữa mass ơi

Original: @Himass gòi gòi
Cleaned : gòi

Original: nói 6 riết
Cleaned : nói 6 riết

Original: đang xem mà còn bị nói xấu nữa mà :))
Cleaned : đang xem mà còn bị nói xấu nữa mà :))

Original: hết giá trụng lợi dị thế thôi
Cleaned : hết giá trụng lợi dị thế thôi

Original: phở ngon chê cơm r
Cleaned : phở ngon chê cơm rồi

Original: Anh pha bảo anh pha bắn hay thì like bằn gà thì cười nha anh em
Cleaned : anh pha bảo anh pha bắn hay thì thích bằn kém thì cười nha anh em

Original: Rồi
Cleaned : rồi

Original: A ban co hay dau ma nhuong a =)))
Cleaned : a cấm co hay dau ma nhuong a =)))

Original: Mai Top 16
Cleaned : mai đường trên 16

Original: Himass bảo, a thấy có vua nào mà nhường cái ngai

In [ ]:
from transformers import AutoModelForMaskedLM, AutoTokenizer
import torch

model= AutoModelForMaskedLM.from_pretrained('uitnlp/visobert')
tokenizer = AutoTokenizer.from_pretrained('uitnlp/visobert')

encoding = tokenizer('hào quang rực rỡ', return_tensors='pt')

with torch.no_grad():
  output = model(**encoding)


In [ ]:
tokenizer.all_special_tokens , tokenizer.all_special_ids

In [ ]:
for word in ["dm", "dcm", "vcl", "vl", "ngu"]:
    tokens = tokenizer.tokenize(word)
    print(word, "->", tokens)

# augmentation

In [ ]:
!pip install nlpaug

In [ ]:
import re
import nlpaug.augmenter.word as naw
import nlpaug.augmenter.char as nac

def augment_with_protection(text, augmenter, special_tokens):
    # Bước 1: Tạo map để bảo vệ
    if text == "":
        return ""
    mapping = {f"SPECIAL_HOLDER_{i}": token for i, token in enumerate(special_tokens)}

    protected_text = text
    for holder, original in mapping.items():
        protected_text = protected_text.replace(original, holder)

    # Bước 2: Augment (Lúc này augmenter thấy SPECIAL_HOLDER_0 nên sẽ ít động vào nếu ta cho vào stopwords)
    # Lưu ý: Cần thêm các holder vào stopwords của augmenter
    augmented_text = augmenter.augment(protected_text)

    # Do nlpaug đôi khi trả về list, ta ép về string
    if isinstance(augmented_text, list):
        augmented_text = augmented_text[0]

    # Bước 3: Trả lại tên cho em
    for holder, original in mapping.items():
        augmented_text = augmented_text.replace(holder, original)

    return augmented_text

# Cấu hình Augmenter
special_tokens = ['<INTENSITY>', '<PROFANITY>', '<LAUGH>' , '<SAD>']
# Để chắc chắn hơn, stopwords nên chứa các holder
holders = [f"SPECIAL_HOLDER_{i}" for i in range(len(special_tokens))]

keyboard_aug = nac.KeyboardAug(aug_char_p=0.1, aug_word_p=0.2, stopwords=holders)
deletion_aug = naw.RandomWordAug(
              action="delete",
              aug_p=0.2, # Xóa 20% số từ,
              stopwords=holders
          )
text = "video này hay quá bạn ơi <LAUGH>"
result = augment_with_protection(text, keyboard_aug, special_tokens)
print(f"Kết quả: {result}")
print(augment_with_protection("" , deletion_aug ,special_tokens ))

# PhoBERT

model general vietnames bert

có head là classifer 2 lớp

(classifier): RobertaClassificationHead(


    (dense): Linear(in_features=768, out_features=768, bias=True)

    (dropout): Dropout(p=0.1, inplace=False)

    (out_proj): Linear(in_features=768, out_features=2, bias=True)


  )

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
model = AutoModelForSequenceClassification.from_pretrained("vinai/phobert-base")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initia

# Phobert-emotion

model pre-trained chuyên với head phân loại 7 lớp


(classifier): RobertaClassificationHead(


    (dense): Linear(in_features=768, out_features=768, bias=True)

    (dropout): Dropout(p=0.1, inplace=False)

    (out_proj): Linear(in_features=768, out_features=7, bias=True)


  )

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("visolex/phobert-emotion")
model = AutoModelForSequenceClassification.from_pretrained("visolex/phobert-emotion")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Load Tokenizer & Model gốc
model_name = "uitnlp/visobert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3) # Ví dụ 3 nhãn

# 2. Định nghĩa danh sách các token đặc biệt bạn đã tạo ra trong pipeline
special_tokens_list = [
    "<INTENSITY>",
    "<PROFANITY>",
    "<INSULT>",
    "<LAUGH>",
    "<SAD>",
]

# 3. Thêm vào Tokenizer
# Việc này gán cho mỗi token một ID riêng biệt trong từ điển
num_added_toks = tokenizer.add_tokens(special_tokens_list)
print(f"Đã thêm {num_added_toks} token mới vào từ điển.")

# 4. QUAN TRỌNG: Resize lại Embedding Matrix của Model
# Vì từ điển to ra, ma trận embedding của model cũng phải to ra tương ứng
model.resize_token_embeddings(len(tokenizer))

# --- TEST THỬ ---
sample_text = "thằng đó ngu <INTENSITY> luôn <LAUGH>"
input_ids = tokenizer.encode(sample_text)
tokens = tokenizer.convert_ids_to_tokens(input_ids)

print("Text gốc:", sample_text)
print("Tokenized:", tokens)

config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/471k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/390M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' me

Đã thêm 5 token mới vào từ điển.
Text gốc: thằng đó ngu <INTENSITY> luôn <LAUGH>
Tokenized: ['<s>', '▁th', 'ằng', '▁đó', '▁ngu', '<INTENSITY>', '▁luôn', '<LAUGH>', '</s>']


# Baselines


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoConfig

class BaseCustomModel(nn.Module):
    """
    Class cơ sở chứa logic đóng băng/mở băng các lớp encoder
    để tránh lặp lại code.
    """
    def _freeze_layers(self, model_backbone, trainable_layers_count):
        # 1. Đóng băng toàn bộ params của backbone trước
        for param in model_backbone.parameters():
            param.requires_grad = False

        # 2. Lấy danh sách các layer encoder (thường nằm trong model.encoder.layer)
        # Cấu trúc của Roberta/Bert thường là: embeddings -> encoder -> pooler
        if hasattr(model_backbone.base_model, 'encoder'):
            layers = model_backbone.base_model.encoder.layer
        else:
            print("Warning: Không tìm thấy cấu trúc encoder chuẩn, không thể freeze theo layer.")
            return

        total_layers = len(layers)

        # 3. Nếu muốn train full thì mở hết
        if trainable_layers_count == -1 or trainable_layers_count >= total_layers:
            for param in model_backbone.parameters():
                param.requires_grad = True
            print(f"-> Đã mở khóa toàn bộ {total_layers} layers của Backbone.")
            return

        # 4. Mở khóa N layers cuối cùng
        if trainable_layers_count > 0:
            start_idx = total_layers - trainable_layers_count
            for i in range(start_idx, total_layers):
                for param in layers[i].parameters():
                    param.requires_grad = True

            # Luôn mở khóa lớp Pooler (nếu có) để train đặc trưng đầu ra
            if hasattr(model_backbone, 'pooler') and model_backbone.pooler is not None:
                for param in model_backbone.pooler.parameters():
                    param.requires_grad = True

            print(f"-> Đã đóng băng {start_idx} layers đầu, chỉ train {trainable_layers_count} layers cuối.")
        else:
            print("-> Đã đóng băng toàn bộ Backbone (Feature Extraction mode).")

# ==============================================================================
# 1. Class PhoBERT (Base)
# ==============================================================================
class PhoBERT_Custom(BaseCustomModel):
    def __init__(self, num_classes=2, trainable_layers=6, dropout_rate=0.1):
        super(PhoBERT_Custom, self).__init__()

        model_name = "vinai/phobert-base"
        self.bert = AutoModel.from_pretrained(model_name)

        # Cấu hình đầu ra của PhoBERT Base là 768
        hidden_size = 768

        # Tùy chỉnh số layer encoder được train
        self._freeze_layers(self.bert, trainable_layers)

        # Custom Head: 2 lớp Dense 768 -> Output
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 768),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(768, 768),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(768, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        # Lấy vector đại diện câu (CLS token)
        cls_output = outputs.last_hidden_state[:, 0, :]

        logits = self.classifier(cls_output)
        return logits

# ==============================================================================
# 2. Class PhoBERT-Emotion (Custom)
# ==============================================================================
class PhoBERT_Emotion_Custom(BaseCustomModel):
    def __init__(self, num_classes=7, trainable_layers=6, dropout_rate=0.1):
        super(PhoBERT_Emotion_Custom, self).__init__()

        model_name = "visolex/phobert-emotion"
        self.bert = AutoModel.from_pretrained(model_name)

        hidden_size = 768
        self._freeze_layers(self.bert, trainable_layers)

        # Custom Head: 2 Dense Layers
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 768),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(768, 768),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(768, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_output)
        return logits

# ==============================================================================
# 3. Class ViSoBERT (Custom from MLM)
# ==============================================================================
class ViSoBERT_Custom(BaseCustomModel):
    def __init__(self, num_classes=2, trainable_layers=6, dropout_rate=0.1):
        super(ViSoBERT_Custom, self).__init__()

        model_name = "uitnlp/visobert"
        self.bert = AutoModel.from_pretrained(model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        hidden_size = 768
        self._freeze_layers(self.bert, trainable_layers)

        # Custom Head: 2 Dense Layers
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 768),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(768, 768),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(768, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_output)
        return logits

# Our model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoConfig

class AttentionMechanism(nn.Module):
    """
    Cơ chế Attention dựa trên bài báo:
    Sử dụng Tanh và Softmax để tính trọng số cho từng bước thời gian.
    Công thức tham khảo: e = tanh(W*h + b) -> a = softmax(e)
    """
    def __init__(self, in_features):
        super(AttentionMechanism, self).__init__()
        # W trong công thức (Project từ hidden_dim -> 1)
        self.attention_score = nn.Linear(in_features, 1)

    def forward(self, hiddens):
        # hiddens shape: [Batch, Seq_Len, Hidden_Dim]

        # 1. Tính điểm e = tanh(W*h + b)
        scores = torch.tanh(self.attention_score(hiddens)) # [Batch, Seq, 1]

        # 2. Tính trọng số a = softmax(e)
        # Softmax trên trục thời gian (dim=1)
        attention_weights = F.softmax(scores, dim=1) # [Batch, Seq, 1]

        # 3. Áp dụng trọng số vào chuỗi đầu vào (Weighting)
        # Thay vì cộng gộp (sum), ta nhân để giữ chuỗi cho CNN xử lý
        context_vector = hiddens * attention_weights # [Batch, Seq, Hidden_Dim]

        return context_vector, attention_weights

class FocalLoss(nn.Module):
    """
    Focal Loss xử lý mất cân bằng dữ liệu (thường gặp trong social media).
    FL(p_t) = -alpha * (1 - p_t)^gamma * log(p_t)
    """
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        BCE_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss) # Lấy xác suất dự đoán đúng
        F_loss = self.alpha * (1-pt)**self.gamma * BCE_loss

        if self.reduction == 'mean':
            return torch.mean(F_loss)
        elif self.reduction == 'sum':
            return torch.sum(F_loss)
        else:
            return F_loss

class VietnameseSocialMediaClassifier(BaseCustomModel):
    def __init__(
        self,
        model_name='uitnlp/visobert',
        num_classes=2,
        trainable_layers = 6,
        lstm_hidden_dim=128,
        cnn_filters=128,
        cnn_kernel_size=3,
        dropout_rate=0.2,
        is_lstm_allowed=True,
        is_attention_allowed=True,
        is_cnn_allowed=True
    ):
        super(VietnameseSocialMediaClassifier, self).__init__()

        # Flags điều khiển kiến trúc (Ablation Study)
        self.is_lstm_allowed = is_lstm_allowed
        self.is_attention_allowed = is_attention_allowed
        self.is_cnn_allowed = is_cnn_allowed

        # 1. Domain-Adaptive Pre-training MLM (ViSoBERT)
        # Sử dụng AutoModel để lấy hidden states
        self.bert = AutoModel.from_pretrained(model_name)

        self._freeze_layers(self.bert, trainable_layers_count=trainable_layers)

        self.bert_hidden_size = self.bert.config.hidden_size # 768 với VisoBERT

        current_dim = self.bert_hidden_size

        # 2. BiLSTM Layer
        if self.is_lstm_allowed:
            self.bilstm = nn.LSTM(
                input_size=current_dim,
                hidden_size=lstm_hidden_dim,
                num_layers=1,
                bidirectional=True,
                batch_first=True
            )
            current_dim = lstm_hidden_dim * 2 # Bidirectional nhân đôi output

        # 3. Attention Mechanism
        if self.is_attention_allowed:
            self.attention = AttentionMechanism(current_dim)
            # Output dimension không đổi sau khi nhân trọng số

        # 4. F. CNN Layer + Max Pooling
        if self.is_cnn_allowed:
            self.cnn = nn.Conv1d(
                in_channels=current_dim,
                out_channels=cnn_filters,
                kernel_size=cnn_kernel_size,
                padding=1 # Giữ kích thước chuỗi
            )
            classifier_input_dim = cnn_filters * 2
        else:
             classifier_input_dim = current_dim

        # 5. Dense Layer (Classifier)
        self.dropout = nn.Dropout(dropout_rate)

        self.classifier = nn.Sequential(
            # Lớp 1: Từ input dim lên 512
            nn.Linear(classifier_input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            # Lớp 2:
            nn.Linear(256, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            # Lớp 3:
            nn.Linear(256, num_classes)
        )
    def forward(self, input_ids, attention_mask):
        # --- 1. ViSoBERT ---
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        x = outputs.last_hidden_state # [Batch, Seq_Len, 768]

        # --- 2. BiLSTM ---
        if self.is_lstm_allowed:
            x, _ = self.bilstm(x) # [Batch, Seq_Len, lstm_hidden*2]

        # --- 3. Attention ---
        if self.is_attention_allowed:
            # Cơ chế Attention bạn đã viết trả về context_vector (chuỗi đã nhân trọng số)
            x, weights = self.attention(x)

        # --- 4. CNN + Hybrid Pooling ---
        if self.is_cnn_allowed:
            # Chuyển về [Batch, Dim, Seq_Len] cho CNN 1D
            x = x.permute(0, 2, 1)
            x = F.relu(self.cnn(x)) # [Batch, Filters, Seq_Len]

            # Global Average Pooling & Global Max Pooling
            avg_pool = F.avg_pool1d(x, kernel_size=x.shape[2]).squeeze(-1)
            max_pool = F.max_pool1d(x, kernel_size=x.shape[2]).squeeze(-1)

            # Kết hợp thông tin: Tổng hợp đặc trưng trung bình và đặc trưng nổi bật nhất
            x = torch.cat((avg_pool, max_pool), dim=1) # [Batch, Filters * 2]
        else:
            # Nếu không có CNN, pooling thủ công trên chiều Seq_Len
            x = torch.mean(x, dim=1) # [Batch, Dim]

        logits = self.classifier(x)

        return logits

# dataset

1. UIT-VSMEC — Vietnamese Social Media Emotion Corpus

👉 Task: Nhận dạng cảm xúc (emotion recognition)

Bộ dữ liệu gồm ~6,927 câu bình luận tiếng Việt được gán nhãn cảm xúc (e.g., enjoyment, sadness, anger, fear, disgust, surprise, other).

Có thể tải bằng thư viện datasets (Hugging Face):

In [ ]:
from datasets import load_dataset
vsmec  = load_dataset("visolex/UIT-VSMEC")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

VSMEC.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/6927 [00:00<?, ? examples/s]

In [ ]:
vsmec = vsmec.remove_columns(["dataset", "type"])

In [ ]:
vsmec

DatasetDict({
    train: Dataset({
        features: ['Sentence', 'Emotion'],
        num_rows: 6927
    })
})

In [ ]:
vsmec['train'].train_test_split(test_size = 0.2)

DatasetDict({
    train: Dataset({
        features: ['Sentence', 'Emotion'],
        num_rows: 5541
    })
    test: Dataset({
        features: ['Sentence', 'Emotion'],
        num_rows: 1386
    })
})

SA-VLSP2016 — Vietnamese Sentiment Analysis

👉 Task: Phân tích cảm xúc (sentiment analysis)

Đây là bộ dữ liệu tiêu chuẩn từ VLSP Shared Task 2016 với nhãn positive / negative / neutral.

Dữ liệu được quản lý bởi tổ chức VLSP; để download bạn phải điền form và gửi email theo hướng dẫn trên trang VLSP.

Bản sao có thể tải trên Hugging Face (dưới điều kiện trích dẫn) như sau:

In [ ]:
from datasets import load_dataset
vlsp_sa  = load_dataset("ura-hcmut/vlsp2016")

README.md:   0%|          | 0.00/286 [00:00<?, ?B/s]

vlsp2016_train.csv: 0.00B [00:00, ?B/s]

vlsp2016_test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/5100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1050 [00:00<?, ? examples/s]

In [ ]:
vlsp_sa

DatasetDict({
    train: Dataset({
        features: ['Class', 'Data'],
        num_rows: 5100
    })
    test: Dataset({
        features: ['Class', 'Data'],
        num_rows: 1050
    })
})

In [ ]:
vlsp_sa.rename_columns({
    'Class' : 'label',
    'Data' : 'text'
 })

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 5100
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 1050
    })
})

In [ ]:
vlsp_sa['train'].to_pandas()['Class'].unique()

array(['negative', 'positive', 'neutral'], dtype=object)

In [ ]:
import pandas as pd

df = pd.read_csv('/content/streaming_data_expanded.csv')
df = df.drop_duplicates(subset=['message'])
df = df[df['emotion'] != 'viewer_request']
df.head()

,message,emotion,toxicity,interaction_type
0,Hello e chè,Neutral,Clean,reaction
1,ddaauf,Neutral,Clean,other
2,zoooo,Excitement,Clean,reaction
3,ô,Neutral,Clean,reaction
4,lo,Neutral,Clean,reaction


In [ ]:
df['emotion'].value_counts()

,count
emotion,
Neutral,17299
Excitement,5778
Frustration,3742
Disappointment,809


In [ ]:
df['toxicity'].value_counts()

,count
toxicity,
Clean,26379
Playful_Toxic,1185
Aggressive_Toxic,64


In [ ]:
df['interaction_type'].value_counts()

,count
interaction_type,
reaction,16391
viewer_request,5395
performance_feedback,3073
other,2648
technical_issue,121


In [ ]:
df.shape

(27628, 4)

In [ ]:
df = df.iloc[:13000]

In [ ]:
df.shape

(13000, 4)

dataset multi-class label classifcation

emotion	toxicity	interaction_type

 -> tập trung vào emotion

# end - to - end pipeline

nhận data -> kết quả độ đo

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from tqdm import tqdm
from datasets import load_dataset


# --- 1. UTILS: DATASET & FOCAL LOSS ---
class SocialMediaDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = cleaning_pipeline(str(self.texts[item]))

        label = self.labels[item]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# --- 2. DATA NORMALIZATION ---
def normalize_format_data(dataset_name, my_custom_df=None):
    """
    Chuẩn hóa mọi nguồn dữ liệu về dạng DataFrame có 2 cột: ['text', 'label']
    """
    df_train = pd.DataFrame()
    df_test = pd.DataFrame()

    print(f"--- Đang xử lý dữ liệu: {dataset_name} ---")

    if dataset_name == 'vlsp2016':
        try:
            df_full = load_dataset("ura-hcmut/vlsp2016")
            df_full = df_full.rename_columns({
                'Class' : 'label',
                'Data' : 'text'
            })

            df_train = df_full['train'].to_pandas()
            df_test = df_full['test'].to_pandas()

        except:
            print("(!) Chưa có file VLSP, tạo dữ liệu mẫu test code.")
            df_train = pd.DataFrame({'text': ['hàng ngon', 'dở tệ'], 'label': ['POS', 'NEG']})
            df_test = pd.DataFrame({'text': ['tạm được'], 'label': ['NEU']})

    elif dataset_name == 'VSMEC':
        try:
            df_full = load_dataset("visolex/UIT-VSMEC")
            df_full = df_full.remove_columns(["dataset", "type"])
            df_full = df_full.rename_columns({
                'Emotion' : 'label',
                'Sentence' : 'text'
            })

            split = df_full['train'].train_test_split(test_size=0.2)

            train = split["train"]
            test  = split["test"]

            df_train = train.to_pandas()
            df_test  = test.to_pandas()
        except:
             print("(!) Chưa có file VSMEC, tạo dữ liệu mẫu test code.")
             df_train = pd.DataFrame({'text': ['buồn quá', 'vui ghê'], 'label': ['Sadness', 'Joy']})
             df_test = pd.DataFrame({'text': ['bình thường'], 'label': ['Other']})

    else:
        if my_custom_df is not None:
            ds = my_custom_df.copy()
            # Xử lý theo yêu cầu của bạn
            if 'toxicity' in ds.columns:
                ds.drop(columns=['toxicity', 'interaction_type'], inplace=True, errors='ignore')

            ds.drop_duplicates(inplace=True)
            ds.reset_index(drop=True, inplace=True)

            if 'emotion' in ds.columns: ds.rename(columns={'emotion': 'label'}, inplace=True)
            if 'message' in ds.columns: ds.rename(columns={'message': 'text'}, inplace=True)

            train, test = train_test_split(ds, test_size=0.2, random_state=42)
            df_train, df_test = train, test

        else:
            print("(!) Lỗi: Chọn 'my_data' nhưng không truyền DataFrame vào.")
            return None, None, None

    le = LabelEncoder()
    all_labels = pd.concat([df_train['label'], df_test['label']]).unique()
    le.fit(all_labels)

    df_train['label'] = le.transform(df_train['label'])
    df_test['label'] = le.transform(df_test['label'])
    print('data : ' , df_train.head())
    print(f"Classes map: {dict(zip(le.classes_, le.transform(le.classes_)))}")

    return df_train, df_test, le

In [ ]:
df_train, df_test, label_encoder = normalize_format_data('my_data', my_custom_df = df)

--- Đang xử lý dữ liệu: my_data ---
data :                                text  label
1965  like cho abe đi ae like like      3
1670                        dế hảo      3
1320                          clmm      2
1018           anh Độ syt luận =))      1
3153                xBuom goat tpp      1
Classes map: {'Disappointment': np.int64(0), 'Excitement': np.int64(1), 'Frustration': np.int64(2), 'Neutral': np.int64(3)}


In [ ]:
df_train, df_test, label_encoder = normalize_format_data('vlsp2016')

--- Đang xử lý dữ liệu: vlsp2016 ---
data :     label                                               text
0      0  Mình đã dùng anywhere thế hệ đầu, quả là đầy t...
1      0  Quan tâm nhất là độ trễ có cao không, dùng thi...
2      0  dag xài con cùi bắp 98k....pin trâu, mỗi tội đ...
3      0  logitech chắc hàng phải tiền triệu trở lên dùn...
4      0  Đang xài con m175 cùi mía , nhà xài nhiều chuộ...
Classes map: {'negative': np.int64(0), 'neutral': np.int64(1), 'positive': np.int64(2)}


In [ ]:
df_train, df_test, label_encoder = normalize_format_data('VSMEC')

--- Đang xử lý dữ liệu: VSMEC ---
data :                                                  text  label
0              vcb làm việc kiểu hách dịch thì có :v      1
1  hãy xử tù chung thân bọn người mất nhân tính g...      0
2  ồ ra thế . tớ sẽ lưu ý :)) mong sẽ có cơ hội s...      6
3                         haha tao trúng số thật này      6
4                             cứu tao :(( tao sợ rắn      3
Classes map: {'Anger': np.int64(0), 'Disgust': np.int64(1), 'Enjoyment': np.int64(2), 'Fear': np.int64(3), 'Other': np.int64(4), 'Sadness': np.int64(5), 'Surprise': np.int64(6)}


In [ ]:
train_dataset = SocialMediaDataset(df_train.text.to_numpy(), df_train.label.to_numpy(), tokenizer)
test_dataset = SocialMediaDataset(df_test.text.to_numpy(), df_test.label.to_numpy(), tokenizer)

In [ ]:
train_dataset[0]

{'text': '<INTENSITY> làm việc kiểu hách dịch thì có vậy',
 'input_ids': tensor([    0, 15002,   129,    15,   542,    17,   287,    13,   391,    22,
           407,    61,    41,    15,   179,     2,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,  

# training

In [ ]:
# --- 3. TRAINING & EVALUATION PIPELINE ---
def train_and_evaluate(dataset_name, model_list, custom_df=None):
    save_models = []
    # 1. Chuẩn bị dữ liệu
    df_train, df_test, label_encoder = normalize_format_data(dataset_name, custom_df)
    num_classes = len(label_encoder.classes_)
    print(f'number of classes : {num_classes}')

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    SPECIAL_TOKENS = [
        "<INTENSITY>", "<PROFANITY>",
        "<LAUGH>", "<SAD>",
    ]

    # Dictionary để lưu kết quả báo cáo
    scientific_report = []

    for model_name in model_list:
        print(f"\n{'='*20} TRAINING MODEL: {model_name} ON {dataset_name} {'='*20}")

        # --- A. KHỞI TẠO MODEL & TOKENIZER ---
        tokenizer = None
        model = None

        # Logic map tên model string -> class object
        if model_name == 'PhoBert':
            tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
            model = PhoBERT_Custom(num_classes=num_classes, trainable_layers = -1).to(device)

        elif model_name == 'Phobert-emotion':
            tokenizer = AutoTokenizer.from_pretrained("visolex/phobert-emotion")
            model = PhoBERT_Emotion_Custom(num_classes=num_classes, trainable_layers = -1).to(device)

        elif model_name == 'visobert':
            tokenizer = AutoTokenizer.from_pretrained("uitnlp/visobert")
            model = ViSoBERT_Custom(num_classes=num_classes ,  trainable_layers = -1).to(device)

        elif model_name == 'Ours':
            tokenizer = AutoTokenizer.from_pretrained("uitnlp/visobert")
            # Giả sử VietnameseSocialMediaClassifier là model hoàn chỉnh của bạn
            model = VietnameseSocialMediaClassifier(
                      num_classes=num_classes, trainable_layers = -1,
                      is_lstm_allowed=True,
                      is_attention_allowed=True,
                      is_cnn_allowed=True
                      ).to(device)



        # ======================================================================
        # [QUAN TRỌNG] THÊM SPECIAL TOKENS & RESIZE EMBEDDING TẠI ĐÂY
        # ======================================================================
        if tokenizer is not None and model is not None:
            # 1. Thêm token vào tokenizer
            num_added_toks = tokenizer.add_tokens(SPECIAL_TOKENS)
            print(f"-> Đã thêm {num_added_toks} token đặc biệt vào Tokenizer.")

            # 2. Resize Embedding của Model để khớp với Tokenizer mới
            # Vì model của bạn là wrapper class (PhoBERT_Custom), ta phải truy cập vào self.bert
            if hasattr(model, 'bert'):
                model.bert.resize_token_embeddings(len(tokenizer))
                print(f"-> Đã resize model.bert embeddings lên: {len(tokenizer)}")
            else:
                try:
                    # Nếu là class VietnameseSocialMediaClassifier (Ours) ở code trước bạn đặt tên là self.bert
                    model.bert.resize_token_embeddings(len(tokenizer))
                    print(f"-> Đã resize Ours model embeddings lên: {len(tokenizer)}")
                except AttributeError:
                    print("(!) Cảnh báo: Không tìm thấy layer embedding để resize.")
        # ======================================================================


        # --- B. DATA LOADERS ---
        train_dataset = SocialMediaDataset(df_train.text.to_numpy(), df_train.label.to_numpy(), tokenizer)
        test_dataset = SocialMediaDataset(df_test.text.to_numpy(), df_test.label.to_numpy(), tokenizer)

        train_loader = DataLoader(train_dataset, batch_size=300, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=300, shuffle=False)

        # --- C. OPTIMIZER & LOSS ---
        optimizer = AdamW(model.parameters(), lr=2e-5)
        criterion = nn.CrossEntropyLoss()
        if model_name == 'Ours':
          criterion = FocalLoss(alpha=0.2, gamma=2.0)

        # --- D. TRAINING LOOP (Rút gọn cho 1 Epoch để demo) ---
        EPOCHS = 6
        best_f1 = 0

        for epoch in range(EPOCHS):
            model.train()
            total_loss = 0
            # Progress bar
            pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}', unit='batch')

            for batch in pbar:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                model.zero_grad()

                # Forward
                outputs = model(input_ids, attention_mask)
                loss = criterion(outputs, labels)

                # Backward
                loss.backward()
                optimizer.step()

                total_loss += loss.item()
                pbar.set_postfix({'loss': total_loss / len(train_loader)})

        # --- E. EVALUATION ---
        model.eval()
        predictions = []
        real_values = []

        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                outputs = model(input_ids, attention_mask)
                _, preds = torch.max(outputs, dim=1)

                predictions.extend(preds.cpu().tolist())
                real_values.extend(labels.cpu().tolist())

        # --- F. METRICS CALCULATION ---
        acc = accuracy_score(real_values, predictions)
        f1_macro = f1_score(real_values, predictions, average='macro')
        f1_weighted = f1_score(real_values, predictions, average='weighted')

        print(f"\n-> [Result] {model_name}: Acc={acc:.4f}, F1-Macro={f1_macro:.4f}")
        print("Classification Report:\n")
        print(classification_report(real_values, predictions,  labels=range(len(label_encoder.classes_)),  target_names=label_encoder.classes_))


        # Lưu kết quả
        save_models.append((tokenizer, model))
        scientific_report.append({
            'Dataset': dataset_name,
            'Model': model_name,
            'Accuracy': round(acc * 100, 2),
            'F1-Macro': round(f1_macro * 100, 2),
            'F1-Weighted': round(f1_weighted * 100, 2)
        })

    return pd.DataFrame(scientific_report) , save_models

# chạy data vlsp2016

# chạy my data

In [ ]:
# 1. Setup Models
models_to_run = ['Ours', 'visobert', 'PhoBert']

# 3. Chạy thực nghiệm
# --- Dataset 1: My Data ---
results_mydata = train_and_evaluate('my_data', models_to_run, custom_df=df)

# 4. Gộp kết quả
final_report = pd.concat([results_vlsp], ignore_index=True)

# 5. Hiển thị bảng kết quả đẹp (Markdown format cho bài báo)
print("\n\n" + "="*40)
print("FINAL SCIENTIFIC REPORT TABLE")
print("="*40)
print(final_report.to_markdown(index=False))

--- Đang xử lý dữ liệu: my_data ---
data :                                                      text  label
19072                  chơi trung thoi cook luôn hả trời      2
8489                               Blv elk à anh phơm:))      3
21917  ?? gà xog làm gì xún KC cho đồ ăn cơm nha abe ...      2
8365                 anh 3 nhìn này =)))) ra sân bay trễ      3
12698                             T1 á anh không bất ngờ      3
Classes map: {'Disappointment': np.int64(0), 'Excitement': np.int64(1), 'Frustration': np.int64(2), 'Neutral': np.int64(3)}
number of classes : 4

==================== TRAINING MODEL: Ours ON my_data ====================


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/471k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/390M [00:00<?, ?B/s]

...

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


-> Đã mở khóa toàn bộ 12 layers của Backbone.
-> Đã thêm 4 token đặc biệt vào Tokenizer.
-> Đã resize model.bert embeddings lên: 64005


Epoch 1/6:   0%|          | 0/74 [00:00<?, ?batch/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Epoch 1/6:  11%|█         | 8/74 [00:11<01:32,  1.40s/batch, loss=0.139]

========================================
FINAL SCIENTIFIC REPORT TABLE
========================================
| Dataset   | Model    |   Accuracy |   F1-Macro |   F1-Weighted |
|:----------|:---------|-----------:|-----------:|--------------:|
| vlsp2016  | Ours     |      0.78 |      0.64 |         0.78 |
| vlsp2016  | visobert |      0.80 |      0.71 |         0.80 |
| vlsp2016  | PhoBert  |      0.84 |      0.74 |         0.85 |